# Détection d'Anomalies IoT Aéronautique (Pipeline Complet ML/XAI)
## Projet de Fin d'Études — Nourhen KHELIFI
### Pipeline Unifié : Vraies Données SurveilDrone-Net23 + XAI

Ce notebook fusionne la préparation des données réelles et l'application des modèles d'intelligence artificielle.

In [ ]:
# Installation des dépendances si nécessaire
# !pip install -q kagglehub shap pandas numpy matplotlib seaborn scikit-learn tensorflow

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import os
import glob
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print("[OK] Imports réussis")

## 1. Extraction et Nettoyage des Vraies Données

In [ ]:
print("Téléchargement du dataset SurveilDrone-Net23...")
path = kagglehub.dataset_download("datasetengineer/surveildrone-net23")
csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)

df = pd.read_csv(csv_files[0])
print(f"Dimensions originales : {df.shape}")

# Conservation des colonnes pertinentes pour notre pipeline
cols_to_keep = ['timestamp', 'ambient_temp_C', 'altitude_m', 'velocity_x', 'velocity_y', 'velocity_z', 
                'battery_level_pct', 'wind_speed_mps', 'acceleration_x', 'acceleration_y', 'acceleration_z', 
                'power_consumption_watts', 'surveillance_pattern']

# Filtrage
df_filtered = df[[c for c in cols_to_keep if c in df.columns]].copy()

# Nettoyage des NaN par interpolation
df_filtered = df_filtered.interpolate(method='linear').fillna(df_filtered.median(numeric_only=True))

# Suppression des doublons
df_filtered = df_filtered.drop_duplicates()

# Nettoyage des incohérences physiques
df_filtered = df_filtered[(df_filtered['battery_level_pct'] >= 0) & (df_filtered['battery_level_pct'] <= 100)]
df_filtered = df_filtered[(df_filtered['ambient_temp_C'] >= -50) & (df_filtered['ambient_temp_C'] <= 150)]

# Conversion du timestamp
df_filtered['timestamp'] = pd.to_datetime(df_filtered['timestamp'])

print(f"Dimensions après nettoyage physique : {df_filtered.shape}")

## 2. Injection d'anomalies (Pour l'évaluation)
Le dataset original n'étant pas labellisé, nous injectons des anomalies synthétiques (vibrations, température, batterie) pour pouvoir évaluer les performances de notre modèle non-supervisé, conformément à la méthodologie du Note 2.

In [ ]:
np.random.seed(42)
N = len(df_filtered)
df_filtered = df_filtered.sort_values('timestamp').reset_index(drop=True)

# Injection de 5% d'anomalies
anomaly_mask = np.zeros(N, dtype=bool)
anom_temp = np.random.choice(N, size=int(N * 0.015), replace=False)
anom_batt = np.random.choice(np.setdiff1d(np.arange(N), anom_temp), size=int(N * 0.012), replace=False)
anom_vib  = np.random.choice(np.setdiff1d(np.arange(N), np.union1d(anom_temp, anom_batt)), size=int(N * 0.018), replace=False)

anomaly_mask[anom_temp] = True
anomaly_mask[anom_batt] = True
anomaly_mask[anom_vib] = True

# Modification des valeurs pour créer l'anomalie physique
df_filtered.loc[anom_temp, 'ambient_temp_C'] += np.random.normal(30, 5, len(anom_temp)) # Surchauffe
df_filtered.loc[anom_batt, 'battery_level_pct'] -= np.random.normal(20, 5, len(anom_batt)) # Chute batterie
df_filtered.loc[anom_vib, 'acceleration_z'] += np.random.normal(5, 2, len(anom_vib)) # Forte vibration

df_filtered['is_anomaly'] = anomaly_mask.astype(int)

print(f"Anomalies injectées : {df_filtered['is_anomaly'].sum()} ({df_filtered['is_anomaly'].mean()*100:.1f}%)")

## 3. Analyse Exploratoire (EDA) sur le dataset unifié

In [ ]:
fig = plt.figure(figsize=(20, 15))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.2)

# --- 1. Pie anomalies ---
ax1 = fig.add_subplot(gs[0, 0])
ac = df_filtered.is_anomaly.value_counts()
ax1.pie([ac[0], ac[1]], labels=['Normal', 'Anomalie'],
        colors=['#4CAF50', '#F44336'], autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax1.set_title('Répartition Normal / Anomalie', fontweight='bold')

# --- 2. Température temporelle ---
ax2 = fig.add_subplot(gs[0, 1])
s = df_filtered.sample(min(2500, len(df_filtered)), random_state=42).sort_values('timestamp')
nm = s.is_anomaly == 0
ax2.scatter(s[nm].timestamp, s[nm].ambient_temp_C, alpha=0.5, s=10, c='#2196F3', label='Normal')
ax2.scatter(s[~nm].timestamp, s[~nm].ambient_temp_C, alpha=0.9, s=40, c='#F44336', marker='^', label='Anomalie')
ax2.set_title('Température - échantillon', fontweight='bold')
ax2.legend(fontsize=10); ax2.tick_params(axis='x', rotation=30)

# --- 3. Heatmap corrélation ---
ax3 = fig.add_subplot(gs[1, :])
num_cols = [c for c in df_filtered.columns if df_filtered[c].dtype in [np.float64, np.int64, np.int32]]
corr = df_filtered[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r',
            center=0, ax=ax3, linewidths=0.5, annot_kws={'size': 9}, vmin=-1, vmax=1)
ax3.set_title('Matrice de corrélation', fontweight='bold')

plt.suptitle('EDA — Dataset Unifié SurveilDrone-Net23', fontsize=16, fontweight='bold')
plt.show()

## 4. Modélisation : Isolation Forest
Entraînement d'un modèle non-supervisé pour détecter les anomalies.

In [ ]:
# Préparation des features
features = ['ambient_temp_C', 'altitude_m', 'velocity_x', 'velocity_y', 'velocity_z', 
            'battery_level_pct', 'wind_speed_mps', 'acceleration_x', 'acceleration_y', 
            'acceleration_z', 'power_consumption_watts']

X = df_filtered[features]
y_true = df_filtered['is_anomaly']

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

# Entraînement Isolation Forest
print("Entraînement de l'Isolation Forest...")
iso_forest = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
preds = iso_forest.fit_predict(X_scaled)

# Conversion des prédictions (-1 = anomalie, 1 = normal) vers (1 = anomalie, 0 = normal)
y_pred = np.where(preds == -1, 1, 0)

print("\nRapport de Classification :")
print(classification_report(y_true, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Anomalie'], yticklabels=['Normal', 'Anomalie'])
plt.title('Matrice de Confusion - Isolation Forest')
plt.ylabel('Vraie Classe')
plt.xlabel('Prédiction')
plt.show()